# page_renderer

> Main management page renderer composing header, toolbar buttons, and document list

In [ ]:
#| default_exp components.page_renderer

In [ ]:
#| export
from typing import Any, Callable, List

from fasthtml.common import Div, H1, A, Span, Button

# DaisyUI components
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import container, max_w, h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, items, justify, gap, grow
)
from cjm_fasthtml_tailwind.utilities.sizing import min_h
from cjm_fasthtml_tailwind.core.base import combine_classes

# Icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V1 button roles, V11 icon-size roles)
from cjm_fasthtml_design_system.buttons import buttons
from cjm_fasthtml_design_system.icons import icons

# Local imports
from cjm_transcript_workflow_management.models import ManagementUrls
from cjm_transcript_workflow_management.html_ids import ManagementHtmlIds
from cjm_transcript_workflow_management.components.import_controls import render_import_controls
from cjm_transcript_workflow_management.components.helpers import DEBUG_MANAGEMENT_RENDER

## Page Header

Title and top-level action buttons (Import, Export All).

In [ ]:
#| export
def render_page_header(
    urls:ManagementUrls,  # URL bundle for route endpoints
) -> Any:  # Header element with title and action buttons
    """Render the management page header with title and top-level actions."""
    return Div(
        # Title
        H1(
            lucide_icon("database", size=icons.page_title),
            "Graph Management",
            cls=combine_classes(
                font_size._3xl, font_weight.bold,
                flex_display, items.center, gap(3)
            )
        ),
        # Action buttons
        Div(
            # Import: toggles import section visibility
            Button(
                lucide_icon("upload", size=icons.text_button),
                "Import",
                cls=combine_classes(
                    buttons.secondary_action,
                    flex_display, items.center, gap(1)
                ),
                onclick=f"document.getElementById('{ManagementHtmlIds.IMPORT_SECTION}').classList.toggle('hidden')",
            ),
            # Export All: plain link for file download
            A(
                lucide_icon("download", size=icons.text_button),
                "Export All",
                cls=combine_classes(
                    buttons.secondary_action,
                    flex_display, items.center, gap(1)
                ),
                href=urls.export_all,
                download=True,
            ),
            cls=combine_classes(flex_display, gap(2)),
        ),
        cls=combine_classes(flex_display, justify.between, items.center, m.b(6)),
    )

In [ ]:
from fasthtml.common import to_xml, Div
from cjm_transcript_workflow_management.models import ManagementUrls

urls = ManagementUrls(
    management_page="/manage/documents/management_page",
    list_documents="/manage/documents/list_documents",
    document_detail="/manage/documents/document_detail",
    delete_document="/manage/documents/delete_document",
    delete_selected="/manage/documents/delete_selected",
    export_document="/manage/export/export_document",
    export_all="/manage/export/export_all",
    import_graph="/manage/import/import_graph",
)

hdr = render_page_header(urls)
xml = to_xml(hdr)
assert "Graph Management" in xml
assert "Import" in xml
assert "Export All" in xml
assert f'href="{urls.export_all}"' in xml
assert 'download' in xml
assert ManagementHtmlIds.IMPORT_SECTION in xml
print("Header OK")

## Management Page

Complete management page composing header and document list.

In [ ]:
#| export
def render_management_page(
    urls:ManagementUrls,  # URL bundle for route endpoints
    render_list_fn:Callable,  # () -> document list component
) -> Any:  # Complete management page component
    """Render the complete management page with header, import section, and document list."""
    if DEBUG_MANAGEMENT_RENDER:
        print(f"[RENDER] management_page")
    
    # Import controls (hidden by default, toggled by Import button)
    import_section = render_import_controls(urls)
    import_section.attrs['cls'] = combine_classes(
        import_section.attrs.get('cls', ''), 'hidden'
    )
    
    return Div(
        render_page_header(urls),
        import_section,
        render_list_fn(),
        id=ManagementHtmlIds.PAGE_CONTENT,
        cls=combine_classes(
            container, max_w._5xl, m.x.auto,
            flex_display, flex_direction.col,
            p(4), gap(4), h.full,
        )
    )

In [ ]:
page = render_management_page(urls, lambda: Div("mock-list", id=ManagementHtmlIds.DOCUMENT_LIST))
xml = to_xml(page)
assert ManagementHtmlIds.PAGE_CONTENT in xml
assert ManagementHtmlIds.DOCUMENT_LIST in xml
assert ManagementHtmlIds.IMPORT_SECTION in xml
assert "hidden" in xml
assert "Graph Management" in xml
assert "mock-list" in xml
print("Management page OK")

In [ ]:
# Test with empty render function
empty_page = render_management_page(urls, lambda: Div("No documents found.", id=ManagementHtmlIds.DOCUMENT_LIST))
xml = to_xml(empty_page)
assert "No documents found." in xml
print("Empty page OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()